# **London Runner**

## **1. Project Setup**

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\candle-fetcher


## **2. Imports**

In [2]:
from src.pipeline import run_london_pipeline, print_pipeline_result
from src.mt5_fetcher import MT5FetchConfig, fetch_raw_candles, print_fetch_result

## **3. Settings**

In [ ]:
# ---------------------------------------------------------------------------
# London Runner Settings
# ---------------------------------------------------------------------------

SYMBOL = "US100.cash"
TIMEFRAME = "1m"

START_DATE = "2021-01-01"
END_DATE = "2021-12-31"

# Optional filters.
# Leave as None if you do not want to use them.
YEARS = None
MONTHS = None
WEEKS = None
DAYS = None
WEEKDAYS = None

OUTPUT_SUFFIX = None
OUTPUT_NAME = None


# ---------------------------------------------------------------------------
# MT5 Fetch Settings
# ---------------------------------------------------------------------------

FETCH_FROM_MT5 = True

RANGE_NAME = "london_range"
RAW_OUTPUT_NAME = None

FETCH_START_DATE = START_DATE
FETCH_END_DATE = END_DATE

CHUNK_DAYS = 31

# Dates are interpreted in this timezone before converting to UTC.
DATE_TIMEZONE_NAME = "Europe/London"

## **4. Fetch Raw Candles From MT5**

In [ ]:
input_pattern = "*.csv"

if FETCH_FROM_MT5:
    fetch_config = MT5FetchConfig(
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        range_name=RANGE_NAME,
        output_name=RAW_OUTPUT_NAME,
        start_date=FETCH_START_DATE,
        end_date=FETCH_END_DATE,
        chunk_days=CHUNK_DAYS,
        date_timezone_name=DATE_TIMEZONE_NAME,
    )

    fetch_result = fetch_raw_candles(fetch_config)
    print_fetch_result(fetch_result)

    # Only process the CSV that was just fetched.
    input_pattern = fetch_result["filename"]

else:
    print("Skipping MT5 fetch. Loading existing CSV files from data/raw/.")

Fetching chunk: 2021-01-01 00:00 -> 2021-02-01 00:00 UTC
Fetching chunk: 2021-02-01 00:00 -> 2021-03-04 00:00 UTC
Fetching chunk: 2021-03-04 00:00 -> 2021-04-04 00:00 UTC
Fetching chunk: 2021-04-04 00:00 -> 2021-05-05 00:00 UTC
Fetching chunk: 2021-05-05 00:00 -> 2021-06-05 00:00 UTC
Fetching chunk: 2021-06-05 00:00 -> 2021-07-06 00:00 UTC
Fetching chunk: 2021-07-06 00:00 -> 2021-08-06 00:00 UTC
Fetching chunk: 2021-08-06 00:00 -> 2021-09-06 00:00 UTC
Fetching chunk: 2021-09-06 00:00 -> 2021-10-07 00:00 UTC
Fetching chunk: 2021-10-07 00:00 -> 2021-11-07 00:00 UTC
Fetching chunk: 2021-11-07 00:00 -> 2021-12-08 00:00 UTC
Fetching chunk: 2021-12-08 00:00 -> 2022-01-01 00:00 UTC
MT5 raw candle fetch complete
Symbol: US100.cash
Timeframe: M1
Rows saved: 109,349
First datetime UTC: 2021-01-04 00:00:00+00:00
Last datetime UTC: 2021-12-31 23:49:00+00:00
Filename: us100_cash_M1_raw_london_range_2021-01-01_to_2021-12-31.csv
Path: C:\candle-fetcher\data\raw\us100_cash_M1_raw_london_range_2021-01-

## **5. Run London Filter**

In [ ]:
result = run_london_pipeline(
    start_date=START_DATE,
    end_date=END_DATE,
    years=YEARS,
    months=MONTHS,
    weeks=WEEKS,
    days=DAYS,
    weekdays=WEEKDAYS,
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    input_pattern=input_pattern,
    output_suffix=OUTPUT_SUFFIX,
    output_name=OUTPUT_NAME,
)

print_pipeline_result(result)

Starting candle pipeline
London: 08:00 to 11:00 (Europe/London)
Loading: C:\candle-fetcher\data\raw\us100_cash_M1_raw_london_range_2021-01-01_to_2021-12-31.csv
Raw rows loaded: 109,349
Raw first datetime: 2021-01-04 00:00:00+00:00
Raw last datetime: 2021-12-31 23:49:00+00:00
Rows after date filters: 109,349
Rows after session filter: 14,393
Export complete
Session: london
Rows saved: 14,393
Filename: us100_cash_M1_london_2021_01_01_to_2021_12_31.csv
Path: C:\candle-fetcher\data\processed\london\us100_cash_M1_london_2021_01_01_to_2021_12_31.csv
Pipeline result
Session: london
Description: London: 08:00 to 11:00 (Europe/London)
Raw rows: 109,349
Date-filtered rows: 109,349
Final rows: 14,393
First datetime: 2021-01-22 08:00:00+00:00
Last datetime: 2021-12-31 10:59:00+00:00
Saved to: C:\candle-fetcher\data\processed\london\us100_cash_M1_london_2021_01_01_to_2021_12_31.csv


## **6. View Data**

In [6]:
df = result["data"]
df.head()

,datetime,open,high,low,close,tick_volume,spread,real_volume,session_name
0,2021-01-22 08:00:00+00:00,13367.55,13372.55,13358.05,13363.30,3766,0,0,london
1,2021-01-22 09:00:00+00:00,13363.55,13369.55,13341.55,13361.80,8793,0,0,london
2,2021-01-22 10:00:00+00:00,13362.55,13364.05,13326.80,13343.55,5695,0,0,london
3,2021-01-25 08:00:00+00:00,13475.40,13496.90,13469.40,13492.90,2741,0,0,london
4,2021-01-25 09:00:00+00:00,13493.15,13493.65,13479.90,13492.15,5556,0,0,london


## **7. Check Size**

In [7]:
df.shape

(14393, 9)